# ESdE Adultos 2023: Data audit and population-representation validation

## Introduction

The scope of this notebook is to provide exploratory analysis of the integrity of the public-use ESdE 2023 adult microdata, with focus on item-level missingness, questionaire routing patterns, and population characterization, for the intention of future explotarion of predictors of health outcomes in following notebooks. 

Its intention is not to reproduce INE's full evaluation of non-response and population representation, which can obtained [here](https://www.ine.es/metodologia/t15/esde23_falta_res.pdf), along the official INE's ESdE [methodology](https://www.ine.es/metodologia/t15/esde23_meto.pdf). 

In [1]:
import pandas as pd
from ine_health_data.pipeline import load_variables, start_setup, is_nonresponse, get_codebook, get_metadata_df
start_setup() 

## Data audit and integrity

### General dataset shape

According to INEs methodology, the number of completed adult questionnaires was 21040 out of 21085 surveyed households.
However in the non-response report, 21077 households were included, with 45 total indicences of non-response.

In the dataset, the total number of rows should represent the number of surveyed adults, which can be contrasted with the household ID `IDENTHOGAR` due to its uniqueness as the sample selects for one adult 15+ whithin each household. 

In [2]:
all_raw_data = load_variables()
pd.Series({
    "n rows": len(all_raw_data),
    "n cols": len(all_raw_data.columns),
    "n IDENTHOGAR": all_raw_data["IDENTHOGAR"].count(),
    "n NaN IDENTHOGAR": all_raw_data["IDENTHOGAR"].isna().sum(),
    "max IDENTHOGAR ID": all_raw_data["IDENTHOGAR"].max(),
    "duplicated IDENTHOGAR": all_raw_data["IDENTHOGAR"].duplicated().sum(),
}).to_frame(name="value")

,value
n rows,21032
n cols,432
n IDENTHOGAR,21032
n NaN IDENTHOGAR,0
max IDENTHOGAR ID,0021077
duplicated IDENTHOGAR,0


The dataset available contains 21032 rows, indicating fewer completed adult questionnaires than reported by methodology document. 
However, based on the non-response document, the count is explained by the 45 incidences of the 21077 household reported, resulting in a total of 21032 surveyed adults.

The discrepancy of 8 records from both INE's official documents is not explained within any of both, and will be ignored due to being out of the scope of this project. 


The amount of columns loaded from the full dataset are in accordance to the number of variables described in the codebook adjacent to the dataset raw file.

### Missingness, item non-response, and questionnaire routing 

As the questionnaire contains conditional response fields that depend on previous answers, it is important to differentiate between routed-out questions, which should be represented with a missing/blank value, and "non-response" or "non-applicable" represented with special code responses.

The table below provides a general overview of non-responses.

In [3]:
total_cells = all_raw_data.size
codebook = get_codebook()

blank_series = all_raw_data.isna().sum()
no_contesta_series = is_nonresponse(df=all_raw_data,nona_labels="No contesta",codebook=codebook).sum()
no_aplicable_series = is_nonresponse(df=all_raw_data,nona_labels="No aplicable",codebook=codebook).sum()
no_consta_series = is_nonresponse(df=all_raw_data,nona_labels="No consta",codebook=codebook).sum()

count = pd.Series({
    "Total cells":total_cells,
    "Blank":blank_series.sum(),
    "'No contesta'": no_contesta_series.sum(),
    "'No aplicable'": no_aplicable_series.sum(),
    "'No consta'": no_consta_series.sum()
})
pct = pd.Series({
    "Total cells": 100,
    "Blank":(blank_series.sum()/total_cells)*100,
    "'No contesta'": (no_contesta_series.sum()/total_cells)*100,
    "'No aplicable'": (no_aplicable_series.sum()/total_cells)*100,
    "'No consta'": (no_consta_series.sum()/total_cells)*100,
}).round(2)
summary = pd.concat({
        "n":count,
        "%":pct
    },axis=1
).sort_values(by="%",ascending=False)

display(summary)
total_non_full = blank_series.sum()+no_contesta_series.sum()+no_aplicable_series.sum()+no_consta_series.sum()
display(f"Total Non-response = {total_non_full}, {(total_non_full/total_cells*100).round(2)}")

,n,%
Total cells,9085824,100.00
Blank,4122552,45.37
'No contesta',45068,0.50
'No consta',2395,0.03
'No aplicable',2165,0.02


'Total Non-response = 4172180, 45.92'

Specific patterns in the dataset per variable group can be seen in the table below.

In [4]:
metadata_df = get_metadata_df(metadata_fields="Grupo")
total_cells_series = pd.Series(
    len(all_raw_data),
    index=all_raw_data.columns,
)
per_group_count = (
    pd.DataFrame({
        "total cells": total_cells_series, 
        "blanks": blank_series, 
        "'No contesta'": no_contesta_series, 
        "'No aplicable'": no_aplicable_series, 
        "'No consta'": no_consta_series
    })
    .join(metadata_df, how="left")
    .groupby("Grupo",sort=False) 
    .sum()
)
per_group_pct = (
    per_group_count
    .div(per_group_count["total cells"], axis=0)
    .mul(100).round(2)
    .add_suffix(" (%)") 
).rename_axis(None)
display(per_group_pct)

,total cells (%),blanks (%),'No contesta' (%),'No aplicable' (%),'No consta' (%)
DATOS DE IDENTIFICACIÓN,100.0,0.00,0.00,0.00,0.00
IDENTIFICACIÓN DEL PROXY,100.0,83.91,0.00,0.00,0.00
A. CARACTERÍSTICAS DEMOGRÁFICAS DE LA PERSONA ADULTA SELECCIONADA,100.0,8.75,1.11,0.00,0.00
B. RELACIÓN DE LA PERSONA ADULTA SELECCIONADA CON LA ACTIVIDAD ECONÓMICA,100.0,73.66,0.34,0.00,0.00
C. ESTADO DE SALUD,100.0,59.66,0.35,0.00,0.00
D. ACCIDENTALIDAD,100.0,23.05,0.17,0.00,0.00
E. AUSENCIA DEL TRABAJO POR PROBLEMAS DE SALUD,100.0,70.48,0.79,0.00,0.00
F. LIMITACIONES FÍSICAS Y SENSORIALES,100.0,5.62,0.19,0.00,0.00
G. LIMITACIONES PARA LA REALIZACIÓN DE LAS ACTIVIDADES DE LA VIDA COTIDIANA,100.0,54.65,0.12,0.27,0.00
H. ESTRÉS Y SATISFACCIÓN LABORAL,100.0,53.05,0.66,0.00,0.00


High number of Blank values are to be expected due to routing-related missingness.

Coded special values are more direct indicators of response status. 

## General population

### Variables data audit

The variable `EDADa` is the only numeric variable loaded, which includes special `999` code for `No contesta` value.
For `SEXOa`, `CCAA`, there are no special coded non-response values.

In [5]:
raw_pop_data = load_variables(variables=["EDADa","SEXOa","CCAA","FACTORADULTO"])
rows_count          = len(raw_pop_data)
nonfull_rows_count  = raw_pop_data.isna().any(axis=1).sum()
age_not_answered_c  = raw_pop_data["EDADa"].eq(999).sum()

pd.Series({
    "Total rows": rows_count,
    "Rows any with empty": nonfull_rows_count,
    "non-response age rows": age_not_answered_c
}).to_frame(name="n")

,n
Total rows,21032
Rows any with empty,0
non-response age rows,0


The dataset for `SEXOa`, `CCAA` and `EDADa` does not contain blanks nor special coded non-response values.
This implication is carried into following python cells, avoiding the need for unnecessary filtering.

### Age and Sex